# Bayesian Machine Learning & Uncertainty Quantification
## Bayesian Inference · MCMC · Bayesian Neural Networks · Conformal Prediction

**TL;DR:** All models are wrong — Bayesian methods tell you HOW wrong. Instead of a single point estimate (θ_MLE), Bayesian inference gives you a full posterior distribution P(θ|data) that represents uncertainty. This matters enormously in biology: an AlphaFold prediction with pLDDT=55 says "I'm not sure" — but how do you calibrate that? This notebook covers Bayesian inference fundamentals, MCMC sampling, Bayesian Neural Networks, and modern uncertainty quantification.

## Bayesian ML — Concepts for Beginners

| Term | Plain English |
|------|---------------|
| **Bayesian inference** | Update beliefs with data: prior (what we knew before) x likelihood (what data says) -> posterior |
| **prior** | Your probability distribution over parameters BEFORE seeing any data |
| **likelihood** | Probability of observing the data given specific parameter values |
| **posterior** | Updated probability distribution over parameters AFTER seeing the data |
| **uncertainty quantification (UQ)** | Measuring how confident the model is in its predictions — essential for high-stakes decisions |
| **aleatoric uncertainty** | Irreducible noise in the data itself (e.g. measurement error) — can't be reduced with more data |
| **epistemic uncertainty** | Uncertainty due to lack of data — CAN be reduced by collecting more examples |
| **MC Dropout** | Run inference N times with dropout active; spread of predictions = uncertainty estimate |
| **conformal prediction** | Statistically rigorous method to build prediction intervals with guaranteed coverage |
| **Gaussian Process (GP)** | Non-parametric model that gives a distribution over functions — exact uncertainty, expensive to scale |
| **MCMC** | Markov Chain Monte Carlo — samples from the posterior when exact computation is intractable |
| **acquisition function** | In Bayesian optimisation: function deciding which point to evaluate next (e.g. Expected Improvement) |

## Beginner Teaching Frame

**Who should fully work through this notebook:** students with stable probability foundations or students willing to take a slower concept-first pass.

**How to study it on a first pass:**
- focus on prior, likelihood, posterior, and uncertainty interpretation
- do not let MCMC mechanics hide the big picture
- connect every uncertainty estimate to a scientific decision

**Common traps:** confusing confidence intervals with credible intervals and treating uncertainty as a side note instead of the main lesson.


## Canonical Resource Upgrade

Use these to strengthen intuition:
- [Think Bayes](https://allendowney.github.io/ThinkBayes2/)
- [Harvard Stat 110](https://stat110.hsites.harvard.edu/)
- [Probabilistic Machine Learning](https://probml.github.io/pml-book/)


## 📄 Primary Literature — Key Papers

These results are based on peer-reviewed publications. Read the originals to go deeper:

- **Gal & Ghahramani 2016** — *Dropout as a Bayesian Approximation (ICML)*  
  [https://arxiv.org/abs/1506.02142](https://arxiv.org/abs/1506.02142)
- **Angelopoulos & Bates 2021** — *A Gentle Introduction to Conformal Prediction*  
  [https://arxiv.org/abs/2107.07511](https://arxiv.org/abs/2107.07511)


## 🗺️ Prerequisites & Learning Path
| | |
|--|--|
| 🔑 Prerequisites | 00/08 Math Foundations (probability, MLE, MAP) |
| 📍 You Are Here | Module 13/01 — Bayesian ML & Uncertainty |
| ➡️ Next Steps | 10/01 Fine-tuning Capstone (uncertainty in protein engineering) |
| 🏁 Goal | Implement MCMC, calibrate a classifier, use conformal prediction for protein function |

### 🆕 From Scratch? Start Here:
1. [3Blue1Brown Bayes Theorem](https://www.youtube.com/watch?v=HZGCoVF3YvM) — 22 min
2. 00/08 Math Foundations — probability section
3. This notebook

**Time:** 12–18 hours | **Difficulty:** ⭐⭐⭐⭐⭐

## Section 1 — Bayesian Inference: From Prior to Posterior

Bayes' rule is simple to write but profound to apply. The **prior** encodes what you believe before seeing data. The **likelihood** encodes what the data tells you. The **posterior** is the update: your belief after seeing the data.

**Conjugate priors** make this analytically tractable: when prior and likelihood combine to give a posterior of the same family, Bayesian updating becomes a simple count increment.

## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Probability basics** — Bayes' theorem, distributions, expected value. *Review: `00_python_ml_basics/08_math_stats_foundations.ipynb`*
- **ML fundamentals** — train/test, loss functions, model selection. *Review: `00_python_ml_basics/02_ml_fundamentals.ipynb`*
- **Neural networks** — for MC Dropout and Bayesian neural networks. *Review: `05_deep_learning_finetuning/01_dl_and_finetuning.ipynb`*

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Bayesian inference: prior → posterior
print("Bayesian Inference: Beta-Binomial Example")
print("=" * 50)

# Scenario: estimating protein binding affinity from experiments
# Prior: Beta(α=2, β=2) — weak prior, slightly biased toward 50%
# Likelihood: Binomial — observed k successes in n trials

alpha_prior, beta_prior = 2, 2  # prior parameters
n_experiments = [10, 50, 200]   # increasing data
k_successes = [7, 38, 160]      # ~70% binding events

fig, axes = plt.subplots(1, len(n_experiments)+1, figsize=(16, 4))

theta = np.linspace(0, 1, 200)
prior_pdf = stats.beta.pdf(theta, alpha_prior, beta_prior)
axes[0].plot(theta, prior_pdf, 'b-', linewidth=2)
axes[0].set_title(f'Prior: Beta({alpha_prior},{beta_prior})')
axes[0].set_xlabel('θ (binding probability)')

for i, (n, k) in enumerate(zip(n_experiments, k_successes)):
    alpha_post = alpha_prior + k
    beta_post  = beta_prior + (n - k)
    posterior = stats.beta.pdf(theta, alpha_post, beta_post)
    post_mean = alpha_post / (alpha_post + beta_post)
    ci_lo, ci_hi = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)

    axes[i+1].plot(theta, prior_pdf, 'b--', alpha=0.4, label='Prior')
    axes[i+1].plot(theta, posterior, 'r-', linewidth=2, label='Posterior')
    axes[i+1].axvline(post_mean, color='r', linestyle=':', label=f'Mean={post_mean:.2f}')
    axes[i+1].set_title(f'n={n}, k={k}: Beta({alpha_post},{beta_post})')
    axes[i+1].set_xlabel('θ')
    axes[i+1].legend(fontsize=7)
    print(f"n={n:3d}, k={k}: posterior mean={post_mean:.3f}, 95% CI=[{ci_lo:.3f},{ci_hi:.3f}]")

plt.tight_layout()
plt.savefig('/tmp/bayesian_update.png', dpi=72)
print("Key: more data → narrower posterior → less uncertainty")

> **Expected output:** Beta-Binomial posterior updates: as n increases, the posterior mean converges and the 95% CI narrows, showing how more data reduces uncertainty  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## Section 2 — MCMC: Sampling from Intractable Posteriors

Most real posteriors cannot be computed analytically. **Markov Chain Monte Carlo** solves this by constructing a Markov chain whose stationary distribution IS the target posterior — then sampling from the chain.

**Metropolis-Hastings** is the fundamental MCMC algorithm. It works even when you only know the target distribution up to a normalizing constant — which is exactly the situation with Bayesian posteriors.

### 📖 Reading Guide — Metropolis-Hastings MCMC Sampling

`current = initial_value`
→ *Start the chain at some initial guess for the parameter.*

`for _ in range(n_samples):`
→ *Run the chain for many steps. Early steps are 'burn-in' — the chain finds the high-probability region.*

`proposed = current + np.random.normal(0, step_size)`
→ *Propose a new value by taking a random step from the current position.*

`log_ratio = log_posterior(proposed) - log_posterior(current)`
→ *Compare the probability of the proposed vs current position. We work in log-space for numerical stability.*

`if log(u) < log_ratio: current = proposed`
→ *Accept the proposal if it's more probable, or sometimes even if it's less probable (this is what lets MCMC explore).*

`samples.append(current)`
→ *Record the current position. After many steps, the collection of positions approximates the posterior distribution.*



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# MCMC: Metropolis-Hastings sampling
print("MCMC: Metropolis-Hastings for Posterior Sampling")
print("=" * 60)

def log_posterior(theta, data, alpha_prior=2, beta_prior=2):
    """Log posterior for Beta-Binomial model."""
    if theta <= 0 or theta >= 1:
        return -np.inf
    k, n = data['k'], data['n']
    log_likelihood = k*np.log(theta) + (n-k)*np.log(1-theta)
    log_prior = (alpha_prior-1)*np.log(theta) + (beta_prior-1)*np.log(1-theta)
    return log_likelihood + log_prior

def metropolis_hastings(log_post, data, n_samples=10000, proposal_std=0.05):
    samples = []
    theta = 0.5  # initial value
    n_accepted = 0

    for _ in range(n_samples):
        # Propose new value
        theta_prop = theta + np.random.normal(0, proposal_std)
        # Accept/reject
        log_ratio = log_post(theta_prop, data) - log_post(theta, data)
        if np.log(np.random.rand()) < log_ratio:
            theta = theta_prop
            n_accepted += 1
        samples.append(theta)

    return np.array(samples), n_accepted / n_samples

np.random.seed(42)
data = {'k': 38, 'n': 50}  # 38 successes in 50 trials
samples, accept_rate = metropolis_hastings(log_posterior, data)

# Discard burn-in
samples = samples[2000:]
print(f"MCMC samples: {len(samples)}")
print(f"Acceptance rate: {accept_rate:.2f} (ideal: 0.2-0.5)")
print(f"Posterior mean: {samples.mean():.3f}")
print(f"Posterior std:  {samples.std():.3f}")
print(f"95% CI: [{np.percentile(samples,2.5):.3f}, {np.percentile(samples,97.5):.3f}]")
print()
# Compare with analytical posterior
alpha_post = 2 + data['k']; beta_post = 2 + data['n'] - data['k']
from scipy import stats
print(f"Analytical: mean={alpha_post/(alpha_post+beta_post):.3f}, "
      f"std={np.sqrt(alpha_post*beta_post/((alpha_post+beta_post)**2*(alpha_post+beta_post+1))):.3f}")

## Section 3 — Conformal Prediction: Model-Agnostic Uncertainty

**Conformal prediction** provides a rigorous, distribution-free uncertainty quantification method with a provable finite-sample coverage guarantee. Unlike Bayesian methods (which require model assumptions) or bootstrapping (which is approximate), conformal prediction guarantees that prediction sets contain the true label with at least 1-α probability — regardless of the underlying model.

This makes it directly applicable to protein function prediction: wrap any model (SVM, XGBoost, ESM2) with conformal prediction and get statistically valid confidence sets.

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# Conformal prediction — model-agnostic uncertainty
print("Conformal Prediction: Guaranteed Coverage")
print("=" * 60)
print()
print("Key property: for any model, conformal prediction gives")
print("P(y ∈ C(x)) ≥ 1 - α with FINITE sample guarantee")
print()
print("Algorithm (split conformal):")
print("  1. Split labeled data: train (fit model) + calibration (compute scores)")
print("  2. Compute nonconformity scores s_i = |y_i - f(x_i)| on calibration")
print("  3. Set threshold q = (1-α) quantile of {s_i}")
print("  4. Prediction set: C(x_new) = {y : |y - f(x_new)| ≤ q}")
print()

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
# Simulate: predict protein ΔΔG from features
n = 500
X = np.random.randn(n, 20)
y = X[:, :5].sum(axis=1) + np.random.randn(n) * 0.5

X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.4, random_state=42)
X_cal, X_test, y_cal, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

# Fit model on training set
sc = StandardScaler()
model = Ridge(alpha=1.0)
model.fit(sc.fit_transform(X_train), y_train)

# Calibration: compute nonconformity scores
y_cal_pred = model.predict(sc.transform(X_cal))
cal_scores = np.abs(y_cal - y_cal_pred)

# Set α = 0.1 (90% coverage)
alpha = 0.1
q = np.quantile(cal_scores, (1-alpha) * (1 + 1/len(cal_scores)))
print(f"Calibration quantile (α={alpha}): q = {q:.3f}")

# Test set coverage
y_test_pred = model.predict(sc.transform(X_test))
in_interval = np.abs(y_test - y_test_pred) <= q
print(f"Test coverage: {in_interval.mean():.3f} (target: {1-alpha:.1f})")
print(f"Average interval width: {2*q:.3f}")
print()
print("Conformal guarantees hold regardless of model class (Ridge, NN, GNN...)")
print("Used in: AlphaFold uncertainty calibration, drug discovery screening")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **Bayesian inference:** Bayes 1763 theorem, de Finetti exchangeability, Lindley paradox
- **MCMC theory:** Detailed balance (reversibility), ergodic Markov chains, burn-in and mixing
- **Variational inference:** ELBO, mean-field approximation, reparameterization trick (connects to VAEs)
- **Conformal prediction:** Coverage guarantee proof, exchangeability assumption, split vs full conformal

### 2️⃣ Must-Have Popular Resources
#### 🟢 Start Here (Zero Bayesian Background)
- 🆓 **3Blue1Brown Bayes Theorem** — [youtube.com/watch?v=HZGCoVF3YvM](https://www.youtube.com/watch?v=HZGCoVF3YvM) — 22 min, essential
- 🆓 **StatQuest Bayesian Statistics** — [youtube.com/c/joshstarmer](https://www.youtube.com/c/joshstarmer) — search "Bayesian" on channel
- 🆓 **Probabilistic ML (Murphy)** — [probml.github.io/pml-book](https://probml.github.io/pml-book/) — free PDF, modern and complete
- 🆓 **PyMC tutorials** — [docs.pymc.io/en/stable/learn](https://www.pymc.io/projects/docs/en/stable/learn.html) — hands-on probabilistic programming
- 📘 **Book:** [Probabilistic ML (Murphy, 2022)](https://probml.github.io/pml-book/) — free PDF, 1000+ pages, modern and comprehensive
- 📘 **Book:** [Bayesian Data Analysis (Gelman)](http://www.stat.columbia.edu/~gelman/book/) — free PDF
- 🎓 **Course:** [Probabilistic ML (Philipp Hennig, Tübingen)](https://www.youtube.com/playlist?list=PL05umP7R6ij2YE8rZv57H2y2EbNPOgiNI) — FREE YouTube
- ⭐ **GitHub:** [pymc-devs/pymc](https://github.com/pymc-devs/pymc) — 9k★ probabilistic programming
- ⭐ **GitHub:** [pytorch/botorch](https://github.com/pytorch/botorch) — 3k★ Bayesian optimization
- 🤗 **HuggingFace:** calibration datasets for model uncertainty evaluation
- 📊 **Kaggle:** [Uncertainty estimation competitions](https://www.kaggle.com/competitions)

### 3️⃣ Practicals — Hands-On
- 💻 Implement Metropolis-Hastings for Bayesian logistic regression, compare to sklearn MLE
- 💻 Use PyMC to fit a hierarchical model to protein thermostability data
- 💻 Implement conformal prediction for protein function classification
- 🔗 GitHub: [pymc-devs/pymc](https://github.com/pymc-devs/pymc) — tutorial notebooks
- 📊 Kaggle: [Bayesian Optimization tutorial notebooks](https://www.kaggle.com/code)

### 4️⃣ Real-World Problems
- 🏭 **Genentech/Roche:** Bayesian clinical trial design (adaptive trials)
- 🏭 **drug discovery companies:** Bayesian optimization for compound screening
- 📊 **Dataset:** [ProteinGym](https://huggingface.co/datasets/OATML-Markslab/ProteinGym) — fitness data for GP-BO
- 🔬 **Application:** Design conformal prediction for variant pathogenicity — provide valid confidence sets

### 5️⃣ Interview Tips
- ❓ **Must know:** Frequentist vs Bayesian — p-value vs credible interval (most common interview question)
- ❓ **Must know:** What is MCMC and why does Metropolis-Hastings sample from the correct distribution?
- ❓ **Must know:** What is conformal prediction's coverage guarantee and what assumption does it require?
- ⚠️ **Common mistake:** Saying "95% CI contains the true parameter with 95% probability" — frequentist CI does NOT work this way
- 💡 **Pro tip:** At Genentech/Roche/AstraZeneca: know the difference between Bayesian and frequentist interpretations of clinical trial results

### 6️⃣ Hot Industry Topics
- 🔥 **Trending:** [Conformal Prediction for LLMs](https://arxiv.org/abs/2309.07255) — valid uncertainty for any foundation model
- 🔥 **Trending:** [BayesOpt for protein engineering](https://github.com/tqchen/xgboost) — BoTorch + ESM2 embeddings
- 🔥 **Trending:** Neural Posterior Estimation (SBI) — likelihood-free Bayesian inference for complex simulators
- 🚀 **Build:** Bayesian optimization loop for protein fitness using BoTorch + ESM2 embeddings
- 🚀 **Build:** Conformal prediction wrapper for AF3 pLDDT — convert to valid confidence sets

## Interview Q&A — Bayesian ML & Uncertainty

**Q: What is the difference between aleatoric and epistemic uncertainty?**
A: Aleatoric: irreducible uncertainty due to noise in data (measurement error, stochastic processes). Can't be reduced with more data. Epistemic: model uncertainty due to limited data — can be reduced by collecting more data. For drug discovery: aleatoric = experimental noise in IC50 measurements; epistemic = uncertainty in poorly-sampled chemical space. Bayesian models capture epistemic uncertainty.

**Q: Why is MCMC needed when we have Bayes' theorem?**
A: Bayes' theorem: P(θ|data) = P(data|θ)·P(θ) / P(data). The denominator P(data) = ∫P(data|θ)P(θ)dθ is an intractable integral in high dimensions (a 100-parameter model requires a 100-dimensional integral). MCMC (Metropolis-Hastings, HMC) samples from the posterior without computing this integral.

**Q: What is conformal prediction and what guarantee does it give?**
A: Conformal prediction wraps ANY model to give prediction sets with a guaranteed coverage: P(Y ∈ C(X)) ≥ 1-α, where α you choose (e.g., 0.05 → 95% coverage). This works for ANY model (neural net, random forest, etc.) and ANY distribution. It requires only a calibration set. For protein design: gives rigorous uncertainty bounds on predicted stability.

**Q: How does a Bayesian neural network differ from a standard neural network?**
A: Standard NN: weights are point estimates. Bayesian NN: weights have distributions — W ~ N(μ_w, σ_w²). Inference: marginalize over weights. Prediction: integrate over weight posterior (approximated by MC dropout, variational inference, or deep ensembles). Deep ensembles (train 5 models with different seeds) are the simplest effective approach.

**Q: When would you use a GP over a deep learning model for protein fitness prediction?**
A: GPs for: <100 labeled measurements (active learning for protein engineering), need calibrated uncertainty for experimental design (Expected Improvement), smooth fitness landscapes (nearby sequences have similar fitness). Deep learning for: >1000 labels, complex non-smooth landscapes, need to leverage sequence context.

### Bayesian Methods — Time Complexity
| Method | Training | Inference | Space |
|--------|----------|-----------|-------|
| Beta-Binomial update | O(1) | O(1) | O(1) |
| MCMC (n steps) | O(n × likelihood) | O(samples) | O(n_params) |
| HMC (L leapfrog) | O(n × L) | O(samples) | O(n_params) |
| GP posterior | O(n³) — Cholesky | O(n²) per point | O(n²) |
| GP with approx. (m inducing) | O(n·m²) | O(m²) | O(m²) |
| Deep ensemble (k models) | O(k × training) | O(k × forward) | O(k × model) |
| Conformal prediction | O(n log n) — sort | O(1) | O(cal set) |

## Mastery Check

Before moving on, make sure you can:
1. explain posterior updating in plain English
2. distinguish a credible interval from a frequentist confidence interval
3. explain why uncertainty matters in biology
4. identify one downstream decision improved by calibrated uncertainty


---
## 📐 Gaussian Process Regression for ΔΔG with Uncertainty

Gaussian Processes give you both a prediction AND calibrated uncertainty — essential for Bayesian optimization of protein fitness.


In [ ]:
# GAUSSIAN PROCESS REGRESSION — Full Implementation
import numpy as np
import matplotlib.pyplot as plt

class GaussianProcess:
    """
    GP regression with RBF kernel.
    Key: returns mean prediction AND variance (uncertainty).
    """
    def __init__(self, length_scale=1.0, signal_var=1.0, noise_var=0.1):
        self.l = length_scale
        self.sigma_f = signal_var
        self.sigma_n = noise_var

    def rbf_kernel(self, X1, X2):
        """K(x,x') = σ_f² * exp(-||x-x'||² / 2l²)"""
        X1 = np.atleast_2d(X1)
        X2 = np.atleast_2d(X2)
        sq_dists = np.sum(X1**2, axis=1)[:, None] + np.sum(X2**2, axis=1)[None, :]                    - 2 * X1 @ X2.T
        return self.sigma_f**2 * np.exp(-0.5 * sq_dists / self.l**2)

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
        K = self.rbf_kernel(X_train, X_train)
        K += (self.sigma_n**2 + 1e-6) * np.eye(len(X_train))
        self.K_inv = np.linalg.inv(K)
        return self

    def predict(self, X_test):
        K_s  = self.rbf_kernel(self.X_train, X_test)    # (n_train, n_test)
        K_ss = self.rbf_kernel(X_test, X_test)            # (n_test, n_test)
        mu = K_s.T @ self.K_inv @ self.y_train
        cov = K_ss - K_s.T @ self.K_inv @ K_s
        var = np.diag(cov)
        return mu, var

# Application: GP for ΔΔG prediction
# Scenario: we have 20 measured mutations, want to predict 100 unmeasured ones
np.random.seed(42)

# True function (unknown, complex)
def true_ddg(x):
    return 0.5 * np.sin(2*x) + 0.2 * x + np.cos(x/2) + 0.1 * x**2 / 10

# Training data (sparse measurements)
n_train = 15
X_train = np.random.uniform(-3, 6, n_train)
y_train = true_ddg(X_train) + np.random.normal(0, 0.3, n_train)

# Test points (dense grid for visualization)
X_test = np.linspace(-3.5, 6.5, 200)

# Fit GP
gp = GaussianProcess(length_scale=1.5, signal_var=1.0, noise_var=0.3)
gp.fit(X_train[:, None], y_train)
mu, var = gp.predict(X_test[:, None])
std = np.sqrt(var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(X_test, true_ddg(X_test), 'k--', linewidth=2, label='True ΔΔG (unknown)', zorder=4)
ax.fill_between(X_test, mu-2*std, mu+2*std, alpha=0.3, color='steelblue', label='95% CI')
ax.fill_between(X_test, mu-std, mu+std, alpha=0.4, color='steelblue', label='68% CI')
ax.plot(X_test, mu, 'b-', linewidth=2, label='GP mean prediction', zorder=3)
ax.scatter(X_train, y_train, s=80, color='red', zorder=5, label='Observed mutations')
ax.set_xlabel('Mutation descriptor (e.g., ESM-2 embedding PC1)')
ax.set_ylabel('ΔΔG (kcal/mol)')
ax.set_title('Gaussian Process ΔΔG Prediction
(uncertainty is widest where data is sparse)')
ax.legend(fontsize=8)

# Where to measure next? Acquisition function: UCB (Upper Confidence Bound)
# Choose the mutation with highest predicted value + exploration bonus
kappa = 2.0  # exploration-exploitation tradeoff
ucb = mu + kappa * std  # UCB acquisition function

ax2 = axes[1]
ax2.plot(X_test, ucb, 'g-', linewidth=2, label=f'UCB (κ={kappa})', zorder=3)
ax2.plot(X_test, mu, 'b--', linewidth=1.5, alpha=0.7, label='GP mean')
ax2.fill_between(X_test, mu, ucb, alpha=0.3, color='green', label='Exploration bonus')
best_next = X_test[np.argmax(ucb)]
ax2.axvline(best_next, color='red', linewidth=2, label=f'Best next point: x={best_next:.2f}')
ax2.set_xlabel('Mutation descriptor')
ax2.set_ylabel('Acquisition value')
ax2.set_title('Bayesian Optimization: Where to Measure Next?
'
              '(balance exploration of uncertain regions with exploitation of known good)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('gp_regression.png', dpi=100)
plt.show()

print("KEY GP CONCEPTS:")
print("  - Prediction = Gaussian distribution, not point estimate")
print("  - Wide CI = high uncertainty (sparse data or extrapolation)")
print("  - Narrow CI = low uncertainty (near training points)")
print("  - UCB acquisition: try regions where µ + kσ is highest")
print()
print("IN PROTEIN DESIGN:")
print("  Round 1: GP fits to 50 initial ΔΔG measurements")
print("  Round 2: UCB selects 20 most informative mutations to synthesize")
print("  Round 3: Update GP with new data, repeat")
print("  → Each round, you learn more efficiently than random sampling")
print()
print("REAL LIBRARY: GPyTorch (GPU-accelerated GP)")
print("  pip install gpytorch")
print("  Handles 10,000+ training points with sparse GP approximations")

In [ ]:
# VARIATIONAL INFERENCE — Scalable Alternative to MCMC
# When MCMC is too slow for neural networks, use VI (mean-field or normalizing flows)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

class BayesianLinear(nn.Module):
    """
    Bayesian linear layer using variational inference.
    Each weight is a Gaussian distribution parameterized by (mu, log_sigma).
    During training: minimize ELBO = E[log p(y|x,w)] - KL[q(w) || p(w)]
    """
    def __init__(self, in_features, out_features, prior_std=1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # Variational parameters: weight mean and log-std
        self.w_mu     = nn.Parameter(torch.zeros(out_features, in_features))
        self.w_log_std = nn.Parameter(torch.full((out_features, in_features), -3.0))
        # Bias parameters
        self.b_mu     = nn.Parameter(torch.zeros(out_features))
        self.b_log_std = nn.Parameter(torch.full((out_features,), -3.0))

    def forward(self, x, n_samples=1):
        outputs = []
        for _ in range(n_samples):
            # Reparameterization trick: w = mu + eps * sigma
            w_std = torch.exp(self.w_log_std)
            b_std = torch.exp(self.b_log_std)
            w = self.w_mu + w_std * torch.randn_like(w_std)
            b = self.b_mu + b_std * torch.randn_like(b_std)
            outputs.append(F.linear(x, w, b))
        return torch.stack(outputs, dim=0).mean(0)

    def kl_divergence(self):
        """KL[N(mu, sigma) || N(0, prior_std²)]"""
        sigma = torch.exp(self.w_log_std)
        kl = 0.5 * (-2*self.w_log_std + sigma**2/self.prior_std**2
                    + self.w_mu**2/self.prior_std**2 - 1).sum()
        return kl

class BayesianMLP(nn.Module):
    def __init__(self, in_dim=1, hidden=32, out_dim=1):
        super().__init__()
        self.l1 = BayesianLinear(in_dim, hidden)
        self.l2 = BayesianLinear(hidden, out_dim)

    def forward(self, x):
        return self.l2(F.relu(self.l1(x)))

    def elbo(self, x, y, n_data, n_samples=5):
        """Evidence Lower Bound: reconstruction - KL divergence"""
        preds = torch.stack([self.forward(x) for _ in range(n_samples)], dim=0)
        recon = -F.mse_loss(preds.mean(0), y)  # likelihood
        kl = (self.l1.kl_divergence() + self.l2.kl_divergence()) / n_data
        return recon - kl, recon, kl

# Train BNN on noisy regression
X = torch.linspace(-3, 6, 100).unsqueeze(-1)
y = (0.5*torch.sin(2*X) + 0.2*X + 0.1*torch.cos(X/2)).squeeze() + torch.randn(100)*0.3
# Only train on subset (simulate sparse data)
train_mask = (X.squeeze() < -1) | (X.squeeze() > 3.5)
X_tr, y_tr = X[train_mask], y[train_mask]

model = BayesianMLP()
opt = torch.optim.Adam(model.parameters(), lr=0.01)

for ep in range(500):
    elbo, recon, kl = model.elbo(X_tr, y_tr, len(X_tr))
    loss = -elbo
    opt.zero_grad(); loss.backward(); opt.step()

# Predict with uncertainty (MC sampling)
model.eval()
with torch.no_grad():
    preds = torch.stack([model(X).squeeze() for _ in range(100)], dim=0)
    mu = preds.mean(0)
    std = preds.std(0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(X.squeeze().numpy(), (mu-2*std).numpy(), (mu+2*std).numpy(),
                alpha=0.25, color='steelblue', label='95% CI (VI)')
ax.plot(X.squeeze().numpy(), mu.numpy(), 'b-', linewidth=2, label='BNN mean')
ax.scatter(X_tr.squeeze().numpy(), y_tr.numpy(), s=30, color='red',
           zorder=5, label='Training data (sparse)')
ax.plot(X.squeeze().numpy(), (0.5*torch.sin(2*X) + 0.2*X + 0.1*torch.cos(X/2)).squeeze().numpy(),
        'k--', linewidth=1.5, label='True function')
ax.axvspan(-1, 3.5, alpha=0.08, color='gray', label='No training data (gap)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Variational Inference BNN — Uncertainty in Data Gaps
'
             '(CI is wide where no training data exists)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('bayesian_vi.png', dpi=100)
plt.show()

print("VI vs MCMC COMPARISON:")
print("  MCMC: exact (in limit), but O(N) per step, can't scale to neural networks")
print("  VI: approximate (assumes q is Gaussian), but O(1) per gradient step")
print("  VI scales to millions of parameters — standard for BNNs")
print()
print("MC DROPOUT (practical BNN shortcut):")
print("  Just leave dropout ACTIVE at test time")
print("  Run N forward passes → ensemble of N slightly different models")
print("  Mean = prediction, Std = uncertainty")
print("  3 lines of code, no architectural changes:")
print("  model.train()  # keep dropout on")
print("  preds = [model(x) for _ in range(50)]")
print("  uncertainty = torch.stack(preds).std(0)")

---
## 📚 Resources — Bayesian ML & Uncertainty Quantification

### University Courses (All Free — Best in the World)
| Course | Institution | Why This One |
|--------|-------------|-------------|
| **[Harvard AM207 Advanced Scientific Computing](https://am207.github.io/2018fall/)** | Harvard | Variational inference, MCMC, Bayesian neural nets. Full lectures + problem sets free. |
| **[Stanford CS228 Probabilistic Graphical Models](https://cs228.stanford.edu/)** | Stanford | Ermon's notes are the best written exposition of PGMs. Free PDF. |
| **[MIT 6.008 Intro to Probability](https://ocw.mit.edu/courses/6-008-introduction-to-inference-spring-2014/)** | MIT | Bayesian inference foundations. Free OCW. |
| **[Harvard STAT110 Probability](https://projects.iq.harvard.edu/stat110/home)** | Harvard | Joe Blitzstein's legendary course. Free YouTube + book. |
| **[Probabilistic ML (Tübingen)](https://github.com/probml/pml-book)** | Tübingen | Two free textbooks by Kevin Murphy (Google Brain). Best modern reference. |

### Essential Reading (Free PDFs)
- **[Probabilistic Machine Learning: An Introduction](https://probml.github.io/pml-book/book1.html)** (Murphy 2022) — Chapters 5, 7, 9, 18: Bayesian reasoning, GPs, neural network uncertainty. Free PDF.
- **[Deep Learning as Approximate Bayesian Inference](https://www.jmlr.org/papers/volume17/14-308/14-308.pdf)** (Gal & Ghahramani 2016) — Monte Carlo Dropout paper; the key justification for dropout at test time
- **[A Tutorial on Gaussian Processes](https://gaussianprocess.org/gpml/chapters/RW.pdf)** (Rasmussen & Williams) — free PDF; chapters 1-2 are sufficient for this module
- **[Variational Inference: A Review](https://arxiv.org/abs/1601.00670)** (Blei et al. 2017) — best survey of VI methods

### Why Uncertainty Matters for Drug Discovery
> At computational biology ML teams and similar companies, a predicted binding affinity of 8.2 nM is useless without a confidence interval. A model that says "8.2 ± 0.1 nM" (certain) drives different experiments than one saying "8.2 ± 5 nM" (uncertain). Bayesian methods are the principled way to quantify this uncertainty.

### Key Concepts
1. **Bayes Theorem**: P(θ|D) ∝ P(D|θ) × P(θ). Posterior = likelihood × prior.
2. **MCMC**: sample from posterior when it's intractable. Metropolis-Hastings, HMC (used in NUTS/Stan).
3. **Variational Inference**: approximate posterior with a tractable family; minimize KL divergence.
4. **Gaussian Processes**: prior over functions; posterior is also a GP (closed form for regression).
5. **MC Dropout**: at test time, keep dropout active; run N forward passes; variance = uncertainty.
6. **Calibration**: a well-calibrated model's 90% confidence interval contains the true value 90% of the time.

### Practice
- **[Harvard AM207 Problem Set 3](https://am207.github.io/2018fall/homeworks/homework3.html)** — Bayesian linear regression + conjugate priors
- **[pyro tutorials](https://pyro.ai/examples/)** — variational inference with a real probabilistic programming language
- **[GPyTorch tutorials](https://gpytorch.ai/)** — Gaussian processes in PyTorch with GPU acceleration


---
## 🎯 Interview Questions — Bayesian ML & Uncertainty

**Q1 (Foundation):** What is the difference between aleatoric and epistemic uncertainty?
> **A:** Aleatoric (data) uncertainty is inherent noise in the data — even with infinite training data, it doesn't decrease. Example: measurement noise in binding affinity assays. Epistemic (model) uncertainty comes from limited data or model misspecification — it decreases as we add more data. Both matter: aleatoric for honest reporting, epistemic for knowing where to collect more data.

**Q2 (Practical):** How does Monte Carlo Dropout approximate Bayesian inference?
> **A:** Gal & Ghahramani (2016) showed that a neural network with dropout is mathematically equivalent to an approximate Bayesian deep Gaussian process. At test time, run N forward passes with dropout ACTIVE (not turned off as usual). The mean of outputs = prediction, variance = uncertainty estimate. Simple to implement, no architecture changes needed.

**Q3 (Calibration):** Your model predicts 95% confidence intervals for ΔΔG values. You check on a holdout set: the intervals contain the true value only 65% of the time. What do you do?
> **A:** The model is overconfident. Apply Platt scaling or temperature scaling to calibrate. On a separate calibration set (not test set!), find a temperature T that when dividing logits by T makes the intervals correctly sized. Or use conformal prediction to get guaranteed coverage without distributional assumptions.

**Q4 (Advanced, computational biology ML teams level):** Why might you prefer a Gaussian Process over a deep neural network for predicting ΔΔG from a small SKEMPI dataset (300 samples)?
> **A:** GPs are principled Bayesian models that naturally give calibrated uncertainty. With 300 samples, a deep network will overfit unless carefully regularized, and its uncertainty estimates via MC Dropout are poorly calibrated. A GP with a protein-structure-aware kernel (e.g., based on ESM-2 embeddings) leverages inductive bias while providing rigorous uncertainty bounds — critical for experimental decision-making with limited data.


---
## 🔗 Cross-Module Connections — How Bayesian Thinking Unifies the Curriculum

### The Bayesian Lens on Every Module

Bayesian reasoning is not just about probability — it's about being explicit about what you don't know.

| Module | Bayesian Connection | What uncertainty looks like |
|--------|--------------------|-----------------------------|
| **07/03** AF3 Confidence | pLDDT is a predicted confidence score — a proxy for uncertainty | pLDDT < 70 = high uncertainty in local structure |
| **12/01** Diffusion | The forward process is a Bayesian posterior; reverse process = prior matching | DDPM ELBO = Bayesian model evidence lower bound |
| **10/01** Fine-Tuning | ΔΔG predictions need calibrated uncertainty; bootstrap = non-parametric Bayes | Overconfident ΔΔG → wrong experimental prioritization |
| **14/01** RL | Thompson Sampling = Bayesian exploration strategy in RL | Uncertainty-guided exploration via GP posterior |
| **15/01** SSL | Self-supervised pre-training learns a prior over protein space | Fine-tuning = Bayesian posterior update given labeled data |
| **16/01** MLOps | Monitoring in production = tracking posterior drift | Distribution shift = posterior no longer matches prior |

### Aleatoric vs Epistemic: The Most Important Distinction

```
Total Uncertainty = Aleatoric (data) + Epistemic (model)

Aleatoric:  irreducible. More data won't help. Inherent noise in the measurement.
            Example: thermal fluctuations in a protein structure (B-factors in PDB)

Epistemic:  reducible. More data will help. Model doesn't know enough yet.
            Example: AF3 pLDDT < 50 on a disordered loop — more structures would help
```

**How to measure each:**
- Aleatoric: heteroscedastic loss — predict mean AND variance (σ²) per output
- Epistemic: ensemble variance, MC Dropout variance, or conformal prediction coverage

### Interview Anchor Question

**Q:** AF3 gives pLDDT = 85 for a residue. Is this aleatoric or epistemic uncertainty?
> **A:** Primarily **epistemic** — AF3 hasn't seen enough similar sequences/structures to be confident. But there's also an aleatoric component: some loops are genuinely disordered in solution (intrinsically disordered regions, or IDRs), and no amount of data will give pLDDT = 95 for them. You can tell the difference by checking if pLDDT correlates with experimental B-factors (aleatoric) or with sequence divergence from training data (epistemic).


## 🐛 Debug Exercise — Bayesian Uncertainty Quantification

Find and fix the **3 bugs** in the uncertainty estimation code below.

**Expected correct output:**
```
MC Dropout produces DIFFERENT predictions each forward pass (stochastic)
Conformal calibration set comes from HELD-OUT data, not training data
Epistemic uncertainty DECREASES with more training data
Aleatoric uncertainty is IRREDUCIBLE noise (data-dependent)
```

<details>
<summary>Show answer</summary>

**Bug 1 — MC Dropout with model.eval():** `model.eval()` disables dropout, making all
forward passes identical. Fix: either keep `model.train()` during inference, or manually
set `p.requires_grad_(False)` and override `Dropout` modules to stay active.
A common pattern: `def enable_dropout(model): [m.train() for m in model.modules() if isinstance(m, nn.Dropout)]`.

**Bug 2 — Conformal calibration from training data:** Calibrating conformal prediction
intervals using training data makes the intervals too narrow (the model fits training data).
Fix: use a separate calibration set that the model has never seen.

**Bug 3 — Epistemic/aleatoric decomposition swapped:** Epistemic uncertainty = variance
of MC sample means across runs (model uncertainty, decreases with data). Aleatoric =
mean of MC sample variances (data noise, irreducible). The code has these swapped.
Fix: `epistemic = predictions.var(axis=0)`, `aleatoric = noise_var` (fixed data noise).
</details>


In [ ]:
# DEBUG EXERCISE — Bayesian uncertainty, find and fix 3 bugs
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

# Simple MLP with dropout for MC Dropout uncertainty estimation
class MCDropoutNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MCDropoutNet()
X_test = torch.linspace(-3, 3, 20).unsqueeze(1)

# BUG 1: model.eval() disables Dropout — all MC samples will be IDENTICAL
model.eval()   # BUG: should keep model.train() or manually enable dropout layers

preds = []
with torch.no_grad():
    for _ in range(50):
        preds.append(model(X_test).squeeze().numpy())

preds = np.array(preds)  # shape [50, 20]
print(f"MC Dropout std (should be > 0 if stochastic): {preds.std(axis=0).mean():.6f}")
print(f"All identical: {np.allclose(preds[0], preds[1])} — BUG: eval() disabled dropout")

# BUG 2: conformal calibration on training data
X_train = np.linspace(-2, 2, 100).reshape(-1, 1)
y_train = np.sin(X_train.ravel()) + np.random.randn(100) * 0.1

# Should use a separate calibration set, never the training set
X_cal, y_cal = X_train, y_train   # BUG: same as training set — intervals will be too tight

# Simulate conformity scores (residuals)
model_simple = lambda x: np.sin(x.ravel())   # oracle model for illustration
cal_scores = np.abs(model_simple(X_cal) - y_cal.ravel())
q_hat = np.quantile(cal_scores, 0.95)
print(f"\nConformal 95% interval half-width (calibrated on train): {q_hat:.4f}")
print("BUG: calibration on training data — intervals are overfit/too narrow")

# BUG 3: epistemic and aleatoric uncertainty swapped
mc_means_per_run = preds   # shape [n_runs, n_points]

# epistemic = variance of means across MC runs (model uncertainty)
# aleatoric = mean variance from data noise (irreducible)
# The code below has them swapped:
aleatoric_wrong = mc_means_per_run.var(axis=0)    # BUG: this is actually EPISTEMIC
epistemic_wrong = np.full(20, 0.01)               # BUG: this is actually ALEATORIC (fixed noise)

print(f"\n'Aleatoric' (actually epistemic, should vary by region): {aleatoric_wrong.mean():.6f}")
print(f"'Epistemic' (actually aleatoric, constant noise): {epistemic_wrong.mean():.6f}")
print("BUG: labels are swapped — variance of MC samples = epistemic, not aleatoric")


## 🐛 Debug Exercise: Metropolis-Hastings Sampler

The MCMC sampler below has 2 bugs that cause it to produce wrong samples. Find and fix them.

In [ ]:
import numpy as np

# BUGGY Metropolis-Hastings sampler
def buggy_mcmc(log_prob_fn, x0, n_samples=5000, step_size=0.5):
    """Sample from a distribution using Metropolis-Hastings."""
    samples = [x0]
    x = x0
    accepted = 0
    for i in range(n_samples):
        # Propose new point
        x_new = x + np.random.normal(0, step_size)
        # Accept/reject
        log_alpha = log_prob_fn(x_new) + log_prob_fn(x)  # BUG 1: should be MINUS, not plus
        if np.log(np.random.uniform()) < log_alpha:
            x = x_new
            accepted += 1
        samples.append(x)
    print(f"Acceptance rate: {accepted/n_samples:.2%}")
    return np.array(samples)  # BUG 2: should discard burn-in (first ~500 samples)

# Target: standard normal N(0,1)
log_normal = lambda x: -0.5 * x**2
samples = buggy_mcmc(log_normal, x0=0.0)
print(f"Mean: {samples.mean():.3f} (should be ~0.0)")
print(f"Std:  {samples.std():.3f} (should be ~1.0)")

# SOLUTION:
# Bug 1: log_alpha = log_prob_fn(x_new) - log_prob_fn(x)  (minus, not plus)
# Bug 2: return np.array(samples[500:])  (discard burn-in)

## 🏋️ Exercise: MC Dropout Uncertainty Estimation

Complete the function that runs a model N times with dropout enabled to estimate prediction uncertainty.

In [ ]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 32)
        self.drop = nn.Dropout(0.3)
        self.fc2 = nn.Linear(32, 1)
    def forward(self, x):
        return self.fc2(self.drop(torch.relu(self.fc1(x))))

def mc_dropout_predict(model, x, n_forward=50):
    """
    Run model n_forward times with dropout active.
    Returns mean prediction and standard deviation (uncertainty).
    
    Args:
        model: trained nn.Module with Dropout layers
        x: input tensor (1, features)
        n_forward: number of forward passes (default 50)
    Returns:
        (mean_pred, std_pred) — mean and uncertainty of predictions
    """
    # TODO: Enable dropout during inference (hint: model.train() keeps dropout active)
    # TODO: Run n_forward passes, collect predictions
    # TODO: Compute mean and std of predictions
    # TODO: Return to eval mode
    pass

model = SimpleNet()
x_test = torch.randn(1, 10)
# mean, std = mc_dropout_predict(model, x_test)
# print(f"Prediction: {mean:.3f} ± {std:.3f}")

# SOLUTION:
# model.train()  # keeps dropout active
# preds = torch.stack([model(x).detach() for _ in range(n_forward)])
# model.eval()
# return preds.mean().item(), preds.std().item()
print("Exercise ready — complete mc_dropout_predict above")

## 🎤 Interview Questions

**Q1 (Easy):** State Bayes' theorem and explain each term in plain English.
<details><summary>Answer</summary>
P(theta|D) = P(D|theta) * P(theta) / P(D). The posterior P(theta|D) is your updated belief about the parameters after seeing data. The likelihood P(D|theta) measures how well the parameters explain the observed data. The prior P(theta) encodes what you believed before seeing data. The evidence P(D) is a normalising constant ensuring the posterior sums to one.
</details>

**Q2 (Easy):** What is the difference between aleatoric and epistemic uncertainty?
<details><summary>Answer</summary>
Aleatoric uncertainty is inherent noise in the data that cannot be reduced by collecting more data (e.g., measurement error in binding assays). Epistemic uncertainty reflects the model's lack of knowledge and decreases as more training data is provided. Distinguishing them matters because epistemic uncertainty signals where active learning should sample next, while aleatoric uncertainty sets the floor on achievable prediction accuracy.
</details>

**Q3 (Easy):** What is MC Dropout and how does it approximate Bayesian inference?
<details><summary>Answer</summary>
MC Dropout keeps dropout active at inference time and runs the same input through the network multiple times, each time with a different random dropout mask. The variation across predictions approximates the posterior predictive distribution. It was shown by Gal and Ghahramani (2016) to be equivalent to variational inference with a specific approximate posterior, making it a practical zero-cost Bayesian approximation.
</details>

**Q4 (Medium):** When would you use conformal prediction instead of MC Dropout?
<details><summary>Answer</summary>
Use conformal prediction when you need distribution-free, finite-sample coverage guarantees regardless of the model architecture. Unlike MC Dropout, conformal prediction makes no assumptions about the model or data distribution — it only requires exchangeability of calibration and test data. It is especially useful in regulated settings (clinical, pharmaceutical) where you must guarantee that prediction intervals contain the true value at least (1-alpha)% of the time, and when the underlying model is not amenable to dropout (e.g., gradient boosting, random forests).
</details>

**Q5 (Medium):** How do you choose a prior in Bayesian ML for protein stability prediction?
<details><summary>Answer</summary>
Start with domain knowledge: experimental DDG values typically range from -5 to +5 kcal/mol, so a Normal(0, 2) prior on the output is reasonable. For model weights, use weakly informative priors like Normal(0, 1) or horseshoe priors for sparsity. If published data exists, use empirical Bayes to set hyperparameters from the marginal distribution of known DDG values. Perform prior predictive checks — sample from the prior and verify the predicted DDG distribution is physically plausible before training.
</details>

**Q6 (Medium):** What is MCMC, and why is it needed when the posterior is intractable?
<details><summary>Answer</summary>
Markov Chain Monte Carlo constructs a Markov chain whose stationary distribution is the target posterior. By drawing correlated samples from this chain (using algorithms like Hamiltonian Monte Carlo or NUTS), you can approximate posterior expectations without computing the intractable normalising constant P(D). MCMC is needed because for most real models, the posterior has no closed-form solution and the parameter space is too high-dimensional for grid-based numerical integration.
</details>

**Q7 (Medium):** Explain Bayesian optimisation and how it applies to protein engineering.
<details><summary>Answer</summary>
Bayesian optimisation uses a surrogate model (typically a Gaussian process) to model the objective function (e.g., protein fitness) and an acquisition function (e.g., Expected Improvement) to decide which sequence to test next. It balances exploration (testing uncertain regions) with exploitation (testing near known good sequences). For protein engineering, this is critical because each wet-lab measurement is expensive and slow — Bayesian optimisation minimises the number of experiments needed to find high-fitness variants by intelligently selecting the most informative mutations to test.
</details>

**Q8 (Hard):** Design an uncertainty-aware protein property predictor and explain how you'd calibrate it.
<details><summary>Answer</summary>
Architecture: ESM-2 backbone with a two-headed output — one for the mean prediction and one for the log-variance (heteroscedastic aleatoric uncertainty). Wrap with MC Dropout (p=0.1) in the prediction heads for epistemic uncertainty. Total uncertainty is the sum of aleatoric variance plus the variance of MC samples. Calibrate using a held-out calibration set: compute the empirical coverage at each confidence level and apply temperature scaling or Platt scaling to the variance estimates. Validate calibration with reliability diagrams (expected vs observed coverage) and the calibration error metric. Deploy conformal prediction on top for guaranteed coverage in production.
</details>

**Q9 (Hard):** Compare Gaussian processes vs deep ensembles for DDG uncertainty quantification.
<details><summary>Answer</summary>
Gaussian processes provide principled uncertainty with exact posterior inference for small datasets (<5000 points) and naturally increase uncertainty far from training data. However, they scale as O(n^3) and struggle with high-dimensional inputs like protein embeddings without kernel approximations. Deep ensembles (5-10 independently trained networks) scale to any dataset size, capture both epistemic uncertainty (disagreement between members) and can model aleatoric uncertainty (per-member variance), but lack theoretical coverage guarantees and are expensive to train. For DDG prediction: use GPs when the labelled dataset is small and inputs are pre-computed fixed-length embeddings; use deep ensembles when fine-tuning the embedding model end-to-end on larger datasets. In practice, deep ensembles often produce better-calibrated uncertainty than single-model Bayesian approximations.
</details>

**Q10 (Hard):** How would you use epistemic uncertainty to guide active learning in a wet-lab feedback loop?
<details><summary>Answer</summary>
Train a DDG predictor with uncertainty estimates (ensemble or MC Dropout) on the initial labelled set. Score all candidate mutations by an acquisition function — pure exploration uses maximum epistemic uncertainty, but Expected Improvement or Upper Confidence Bound balances exploitation. Select a batch of 10-50 mutations maximising batch diversity (using DPP or clustering in embedding space to avoid redundancy). Send to wet-lab for experimental measurement. Retrain the model on the expanded dataset and repeat. Key practical considerations: batch size is constrained by experimental throughput, you should monitor calibration each round, and you may need to account for experimental noise by weighting samples by measurement confidence.
</details>

## Notebook Complete

**You can now:**
- Quantify aleatoric and epistemic uncertainty with MC Dropout and conformal prediction
- Apply Bayesian optimisation with a Gaussian Process surrogate for protein design

**Where this fits:**
- Previous: 12_generative_models
- Next: 14_reinforcement_learning/01_rl_protein_design — Module 14 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes